<a href="https://colab.research.google.com/github/reezyhudson865/AI_Projects/blob/main/RHudson_Qwen_PromptEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 - Challenge Lab: Prompt Engineering

**Course:** AI 301 - Natural Language Processing and Conversational AI
**Student:** Reezy Hudson   


The following notebook follows a lab completed in an undergrad Applied AI class at UT-Knoxville in the Fall 2027 semester. It follows prompt engineering using Qwn. The lab follows a model outlined by the professor. All rights reserved.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import re
from datasets import load_dataset

# ten BANKING77 intents, two for each of our five categories
intent_to_category = {
    "card_not_working": "card_problem",
    "lost_or_stolen_card": "card_problem",
    "pending_transfer": "transfer",
    "failed_transfer": "transfer",
    "top_up_failed": "top_up",
    "pending_top_up": "top_up",
    "pin_blocked": "pin_and_identity",
    "verify_my_identity": "pin_and_identity",
    "transfer_fee_charged": "fees",
    "card_payment_fee_charged": "fees",
}
categories = ["card_problem", "transfer", "top_up", "pin_and_identity", "fees"]

banking = load_dataset("legacy-datasets/banking77")


def take(split):
    # six random messages from each category, the same six for everyone (random_state=0)
    df = banking[split].to_pandas()
    df["intent"] = banking[split].features["label"].int2str(list(df["label"]))
    df = df[df["intent"].isin(intent_to_category)].copy()
    df["category"] = df["intent"].map(intent_to_category)
    df = df.groupby("category").sample(6, random_state=0)
    return list(df["text"]), list(df["category"])


test_texts, test_labels = take("test")
train_texts, train_labels = take("train")

# five GSM8K problems; the number after #### in the answer is the final answer
gsm = load_dataset("openai/gsm8k", "main")["test"].to_pandas().sample(5, random_state=0)
problems = []
for question, solution in zip(gsm["question"], gsm["answer"]):
    final = solution.split("####")[-1].strip().replace(",", "")
    problems.append((question, int(final)))

print("test messages:", len(test_texts), " training messages:", len(train_texts))
print("math problems:", len(problems))

README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  298kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 93.9kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

... (5 outputs left out)

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

test messages: 30  training messages: 30
math problems: 5


In [ ]:
print("BANKING77 before:", len(banking["test"]), "test rows,",
      banking["test"].features["label"].num_classes, "intents")
raw = banking["test"][0]
print(raw)
print("label", raw["label"], "is the intent", banking["test"].features["label"].int2str(raw["label"]))
print()
print("BANKING77 after:", len(test_texts), "test messages, 5 categories")
print(test_labels[0], "|", test_texts[0])
print()
print("GSM8K before, the full answer:")
print(gsm["answer"].iloc[0])
print()
print("GSM8K after:")
print("question:", problems[0][0])
print("answer:", problems[0][1])

BANKING77 before: 3080 test rows, 77 intents
{'text': 'How do I locate my card?', 'label': 11}
label 11 is the intent card_arrival

BANKING77 after: 30 test messages, 5 categories
card_problem | card is lost, please help

GSM8K before, the full answer:
If rice took thirty minutes to cook, pork took 20+30 = <<20+30=50>>50 minutes.
The total cooking time for pork and rice is 50+30 = <<50+30=80>>80 minutes.
The beans cooked 1/2*80 = <<80*1/2=40>>40 minutes to cook.
The total amount of time taken to cook all the food is 40+50+30 = <<40+50+30=120>>120 minutes.
#### 120

GSM8K after:
question: Finley went to the grocery store and bought rice, beans, and pork for use in their home. It took her 20 more minutes to cook pork than rice, while beans took half the combined cooking time of pork and rice. If it took her 30 minutes to cook rice, how long in minutes did it take to cook all the food?
answer: 120


In [ ]:
from sklearn.metrics import accuracy_score, classification_report


def to_category(reply):
    # the first category name that appears in the reply, or "unknown"
    text = reply.lower().replace(" ", "_")
    for category in categories:
        if category in text:
            return category
    return "unknown"


def score(predictions, labels, title=""):
    # print accuracy and the per category report, and return accuracy
    accuracy = accuracy_score(labels, predictions)
    print(title, "accuracy:", round(accuracy, 3))
    print(classification_report(labels, predictions, labels=categories, zero_division=0))
    return accuracy


def last_number(text):
    # the last whole number in the text, or None if there is none
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    if len(numbers) == 0:
        return None
    return int(numbers[-1])

## 1. Instruction tuned model

This is the same instruction tuned model as the guided labs. Every exercise sends it a list of messages
through one helper, `ask`, so every run uses the same decoding settings.

> Model card: [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, set_seed

model_name = "Qwen/Qwen2.5-1.5B-Instruct"


def ask(messages, max_new_tokens=200, do_sample=False, temperature=1.0, top_p=1.0, top_k=0):
    result = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )
    return result[0]['generated_text']


def count_tokens(text):
    return len(tokenizer(text)["input_ids"])


# TODO: "cuda" when a GPU is attached, "mps" on a Mac with Apple silicon, otherwise "cpu"
# hint: torch.cuda.is_available() and torch.backends.mps.is_available()
device = "cuda"
print("device:", device)

# TODO: load the tokenizer and the model by model_name, and move the model to the device
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)


# TODO: a text-generation pipeline that returns only the reply
# hint: Guided Lab Part 2, Part 0, return_full_text
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)


# test: run after your code above
reply = ask([{"role": "user", "content": "In one sentence, what does a bank transfer do?"}], max_new_tokens=60)
if reply is None or count_tokens("test") is None:
    print("Write the device, the model, ask and count_tokens above, then run this cell again.")
else:
    print(reply)
    print("tokens:", count_tokens(reply))

# Expected output (your reply will differ):
#
# device: cuda
# A bank transfer moves money from one account to another ...
# tokens: xx

device: cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A bank transfer moves money from one account to another through the banking system.
tokens: 15


In [ ]:
def classify_all(build, texts):
    # build(message) returns a list of messages; returns one category per text
    predictions = []
    for text in texts:
        reply = ask(build(text), max_new_tokens=15)
        predictions.append(to_category(reply))
    return predictions

In [ ]:
def full_prompt(message):
    # TODO: the persona, the instruction, the context (what each category means)
    # and the output format, as one string
    system = """
    You are a banking customer service assistant.

    Your task is to classify a customer's message
    into one of five categories.

    Categories:
    card_problem: Problems with lost, stolen, or nonworking cards.
    transfer: Problems with pending or failed bank transfers.
    top_up: Problems with pending or failed account top-ups.
    pin_and_identity: Problems with PINs or identity verification.
    fees: Questions about transfer or card payment fees.

    Respond with only the category name.
    Do not include explanations.
    """

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": message},
    ]


def reduced_prompt(message):
    system = """
    You are a banking customer service assistant.

    Your task is to classify a customer's message
    into one of five categories:
    card_problem, transfer, top_up, pin_and_identity, fees.

    Respond with only the category name.
    Do not include explanations.
    """

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": message},
    ]


# test: one message through each prompt before the full runs
if reduced_prompt(test_texts[0]) is None:
    print("Fill in full_prompt and write reduced_prompt above, then run this cell again.")
else:
    print("message:", test_texts[0], "| label:", test_labels[0])
    print("full prompt reply:", ask(full_prompt(test_texts[0]), max_new_tokens=15))
    print("reduced prompt reply:", ask(reduced_prompt(test_texts[0]), max_new_tokens=15))


# TODO: run each prompt on the test messages with classify_all
full_predictions = classify_all(full_prompt, test_texts)
full_accuracy = score(full_predictions, test_labels, "full prompt")
print()
reduced_predictions = classify_all(reduced_prompt, test_texts)
reduced_accuracy = score(reduced_predictions, test_labels, "reduced prompt")

# Expected output (your numbers will differ):
#
# full prompt accuracy: 0.xxx
#                   precision    recall  f1-score   support
#
#     card_problem      0.xxx     0.xxx     0.xxx         6
#         transfer      0.xxx     0.xxx     0.xxx         6
#           top_up      0.xxx     0.xxx     0.xxx         6
# pin_and_identity      0.xxx     0.xxx     0.xxx         6
#             fees      0.xxx     0.xxx     0.xxx         6
# [averages]
#
# reduced prompt accuracy: 0.xxx
# [classification report, laid out as above]

message: card is lost, please help | label: card_problem
full prompt reply: card_problem
reduced prompt reply: card_problem


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


full prompt accuracy: 0.967
                  precision    recall  f1-score   support

    card_problem       0.86      1.00      0.92         6
        transfer       1.00      1.00      1.00         6
          top_up       1.00      1.00      1.00         6
pin_and_identity       1.00      0.83      0.91         6
            fees       1.00      1.00      1.00         6

        accuracy                           0.97        30
       macro avg       0.97      0.97      0.97        30
    weighted avg       0.97      0.97      0.97        30


reduced prompt accuracy: 0.867
                  precision    recall  f1-score   support

    card_problem       0.75      1.00      0.86         6
        transfer       0.75      1.00      0.86         6
          top_up       1.00      0.67      0.80         6
pin_and_identity       1.00      0.83      0.91         6
            fees       1.00      0.83      0.91         6

        accuracy                           0.87        30
       

In [ ]:
for i, (full, reduced) in enumerate(zip(full_predictions, reduced_predictions)):
    if full != reduced:
        print("Message:", test_texts[i])
        print("Correct label:", test_labels[i])
        print("Full prompt:", full)
        print("Reduced prompt:", reduced)
        print()



Message: I had to pay in order to do a transfer
Correct label: fees
Full prompt: fees
Reduced prompt: transfer

Message: What's the deal? My card was just denied for top up. Why is it not going through?
Correct label: top_up
Full prompt: top_up
Reduced prompt: card_problem

Message: Problems with top up
Correct label: top_up
Full prompt: top_up
Reduced prompt: transfer



In [ ]:
def prompt_tokens(messages):
    # tokens in the full prompt after the chat template is applied
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return len(tokenizer(text)["input_ids"])


for i in range(len(train_texts)):
    print(i, train_labels[i], "|", train_texts[i])

0 card_problem | I lost my wallet and all my cards were in it.
1 card_problem | My card is missing.
2 card_problem | How do I re-order a damaged card?
3 card_problem | I tried to purchase airline tickets online this morning and my card information isn't working. Is there something going on with my account?
4 card_problem | My actual card isn't functioning.
5 card_problem | I tried using my card, but it doesn't work.
6 fees | I recently bought an item using my card and I was charged an extra fee why is this?
7 fees | Hi! I'm a university student studying abroad and I noticed that when I was trying to cross-reference my budget for this month that there were extra fees I was not expecting. Why is that? I've had this account for years and I haven't had any trouble so far, it might not even be on your all's end since I'm still new to international travel and finance, but I wanted to double check in case a mistake was made!
8 fees | I think I have an extra transfer charge.
9 fees | why was i

In [ ]:
def few_shot_prompt(message, example_ids):
    messages = [full_prompt(message)[0]]
    for i in example_ids:
        messages.append({"role": "user", "content": train_texts[i]})
        messages.append({"role": "assistant", "content": train_labels[i]})

    messages.append({"role": "user", "content": message})
    return messages

# test: the roles in a prompt with two examples, and one reply
messages = few_shot_prompt(test_texts[0], [0, 6])
if messages is None:
    print("Write few_shot_prompt above, then run this cell again.")
else:
    for m in messages:
        print(m["role"], "|", m["content"][:60])
    print("reply:", ask(messages, max_new_tokens=15), "| label:", test_labels[0])


# TODO: indexes of your examples from the training list, one per category (5 indexes)
one_each = [0, 6, 12, 18, 24]


def one_shot_build(message):
    return few_shot_prompt(message, one_each)


# TODO: run the few-shot prompt on the test messages and score it
few_predictions = classify_all(one_shot_build, test_texts)
score(few_predictions, test_labels, "few-shot")
print()

# TODO: the prompt tokens for the first test message with each prompt
print("prompt tokens, Exercise 2 prompt:", prompt_tokens(full_prompt(test_texts[0])))
print("prompt tokens, few-shot prompt:", prompt_tokens(one_shot_build(test_texts[0])))

# Expected output (your numbers will differ):
#
# few-shot accuracy: 0.xxx
# [classification report, laid out as in Exercise 2]
#
# prompt tokens, Exercise 2 prompt: xxx
# prompt tokens, few-shot prompt: xxx

system | 
    You are a banking customer service assistant.

    Your
user | I lost my wallet and all my cards were in it.
assistant | card_problem
user | I recently bought an item using my card and I was charged an
assistant | fees
user | card is lost, please help
reply: card_problem | label: card_problem
few-shot accuracy: 0.933
                  precision    recall  f1-score   support

    card_problem       0.86      1.00      0.92         6
        transfer       0.86      1.00      0.92         6
          top_up       1.00      1.00      1.00         6
pin_and_identity       1.00      0.83      0.91         6
            fees       1.00      0.83      0.91         6

        accuracy                           0.93        30
       macro avg       0.94      0.93      0.93        30
    weighted avg       0.94      0.93      0.93        30


prompt tokens, Exercise 2 prompt: 127
prompt tokens, few-shot prompt: 276


| Prompt | Accuracy | Prompt tokens |
|---|---|---|
| Full prompt, no examples (Exercise 2) | | |
| Few-shot, one example per category | | |

In [ ]:
def zero_shot_cot(question):
    return [
        {"role": "user", "content": question + " Solve this problem step by step return the final numerical answer."},
    ]


# TODO: a short word problem of your own and a worked answer that shows each step
# hint: the worked answer in Guided Lab Part 2 shows the shape: each step with its arithmetic,
# then a final sentence with the number
example_question = "A student buys 4 notebooks for $3 each and 2 pens for $2 each. How much does the student spend in total?"
example_answer = """
Step 1: Calculate the notebook cost: 4 * 3 = 12.
Step 2: Calculate the pen cost: 2 * 2 = 4.
Step 3: Add both costs: 12 + 4 = 16.
The student spends $16 in total.
"""


def few_shot_cot(question):
    return [
        # TODO: your example question as a user turn
        {'role': 'user', 'content': example_question},
        # TODO: your worked answer as an assistant turn
        {'role': 'assistant', 'content': example_answer},
        {"role": "user", "content": question},
    ]


# test: the first problem with each prompt
question, answer = problems[0]
if zero_shot_cot(question) is None or example_question == "TODO":
    print("Write zero_shot_cot and fill in few_shot_cot above, then run this cell again.")
else:
    print("QUESTION:", question)
    print("CORRECT ANSWER:", answer)
    print()
    reply = ask(zero_shot_cot(question), max_new_tokens=300)
    print("ZERO-SHOT:", reply)
    print("last number:", last_number(reply))
    print()
    reply = ask(few_shot_cot(question), max_new_tokens=300)
    print("FEW-SHOT:", reply)
    print("last number:", last_number(reply))


methods = {"zero-shot chain of thought": zero_shot_cot, "few-shot chain of thought": few_shot_cot}
replies = {}
for name in methods:
    replies[name] = []
    correct = 0
    tokens = 0
    for question, answer in problems:
        # TODO: ask the model with this method's prompt, max_new_tokens=300
        reply = ask(methods[name](question), max_new_tokens=300)
        replies[name].append(reply)

        # TODO: add this reply's tokens, and count it when its last number is the answer
        tokens = tokens + count_tokens(reply)
        if last_number(reply) == answer:
            correct = correct + 1
    print(name + ":", correct, "of", len(problems), "correct,", tokens, "reply tokens")

# print the first problem the few-shot method got wrong
print()
for i in range(len(problems)):
    if last_number(replies["few-shot chain of thought"][i]) != problems[i][1]:
        print("QUESTION:", problems[i][0])
        print("CORRECT ANSWER:", problems[i][1])
        print(replies["few-shot chain of thought"][i])
        break

# Expected output (your numbers will differ):
#
# zero-shot chain of thought: x of 5 correct, xxxx reply tokens
# few-shot chain of thought: x of 5 correct, xxxx reply tokens
#
# QUESTION: ...
# CORRECT ANSWER: xx
# [the full wrong reply]

QUESTION: Finley went to the grocery store and bought rice, beans, and pork for use in their home. It took her 20 more minutes to cook pork than rice, while beans took half the combined cooking time of pork and rice. If it took her 30 minutes to cook rice, how long in minutes did it take to cook all the food?
CORRECT ANSWER: 120

ZERO-SHOT: Let's break down the information given:

1. Finley spent 30 minutes cooking rice.
2. Cooking pork took 20 more minutes than cooking rice.
   - Therefore, cooking pork took \(30 + 20 = 50\) minutes.
3. Cooking beans took half the combined cooking time of pork and rice.
   - Combined cooking time of pork and rice is \(30 + 50 = 80\) minutes.
   - So, cooking beans took \(\frac{80}{2} = 40\) minutes.

Now we can calculate the total time taken to cook all the food:
- Rice: 30 minutes
- Pork: 50 minutes
- Beans: 40 minutes

Total time = \(30 + 50 + 40 = 120\) minutes.

Therefore, it took Finley 120 minutes to cook all the food.
last number: 120

FEW-SHOT

In [ ]:
expected_keys = ["category", "summary", "urgency"]
urgency_levels = ["low", "medium", "high"]


def parse(text):
    # return the dictionary if text is valid JSON, otherwise None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def remove_code_block(text):
    # drop a ```json ... ``` wrapper if the reply has one
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

## ✏️ EXERCISE 5 (15 points)

1. Write `ticket_prompt(message)`, which asks for the ticket and shows the exact JSON format, with
   the allowed category and urgency values.
2. Run it on the first 10 test messages with `max_new_tokens=120`.
3. Each reply goes through four checks: it parses as it is, it parses after `remove_code_block`, it
   has exactly the expected keys, and its `category` and `urgency` are allowed values. Checks 1 to 3
   are done. Write check 4.
4. Print the counts and one reply that failed a check, if any did.

*Reference:* Guided Lab Part 2, Part 5, for the template prompt, `parse` and the key check. The allowed value check is new.

In [ ]:
def ticket_prompt(message):
    return [
        {
            "role": "system",
            "content": """
            You are a banking customer support assistant.
            Convert customer messages into support tickets.

            Choose one category:
            card_problem, transfer, top_up, pin_and_identity, fees.

            Choose one urgency level:
            low, medium, high.

            Return only valid JSON in this exact format:
            {
                "category": "transfer",
                "urgency": "medium",
                "summary": "Customer reports a delayed bank transfer."
            }

            Do not include explanations or markdown.
            """
        },
        {"role": "user", "content": message}
    ]


# test: one message and its parsed ticket
if ticket_prompt(test_texts[0]) is None:
    print("Write ticket_prompt above, then run this cell again.")
else:
    reply = ask(ticket_prompt(test_texts[0]), max_new_tokens=120)
    print(reply)
    print()
    print("parsed after cleanup:", parse(remove_code_block(reply)))


parses = 0
parses_clean = 0
has_keys = 0
allowed = 0
failed = None
for text in test_texts[:10]:
    # TODO: ask for the ticket, max_new_tokens=120
    reply = ask(ticket_prompt(text), max_new_tokens=120)
    ok = True

    # check 1: the reply parses as it is
    if parse(reply) is not None:
        parses = parses + 1

    # check 2: parse the reply after removing a code block
    data = parse(remove_code_block(reply))
    if data is None:
        ok = False
    else:
        parses_clean = parses_clean + 1
        # check 3: a dictionary whose sorted keys equal expected_keys
        if isinstance(data, dict) and sorted(data.keys()) == expected_keys:
            has_keys = has_keys + 1
            # TODO: check 4, category in categories and urgency in urgency_levels
            if data['category'] in categories and data['urgency'] in urgency_levels:
                allowed = allowed + 1
            else:
                ok = False
        else:
            ok = False
    if not ok and failed is None:
        failed = reply

print("parses:", parses, "of 10")
print("parses after cleanup:", parses_clean, "of 10")
print("expected keys:", has_keys, "of 10")
print("allowed values:", allowed, "of 10")
print()
print("FAILED REPLY:")
print(failed)

# Expected output (your numbers will differ):
#
# parses: x of 10
# parses after cleanup: x of 10
# expected keys: x of 10
# allowed values: x of 10
#
# FAILED REPLY:
# ...

{
    "category": "pin_and_identity",
    "urgency": "high",
    "summary": "Card lost and need assistance to replace it"
}

parsed after cleanup: {'category': 'pin_and_identity', 'urgency': 'high', 'summary': 'Card lost and need assistance to replace it'}
parses: 10 of 10
parses after cleanup: 10 of 10
expected keys: 10 of 10
allowed values: 10 of 10

FAILED REPLY:
None


In [ ]:
policy = '''Northside Bank fee policy
Card payments in US dollars are free. Card payments in another currency cost 2% of the payment.
The first 3 ATM withdrawals each month are free. After that, each withdrawal costs $2.50.
Transfers to another Northside account are free and arrive instantly.
Transfers to other banks cost $1 and arrive within 1 to 2 business days.
Top ups by debit card are free. Top ups by credit card cost 1.5%.
A replacement card costs $5. Replacing a stolen card is free if it is reported within 24 hours.'''

questions = [
    ("How much does a card payment in euros cost?", "answerable"),
    ("I already used the ATM three times this month. What will the next withdrawal cost?", "answerable"),
    ("Does it cost anything to top up with a credit card?", "answerable"),
    ("My card was stolen this morning. Do I pay for the new card?", "answerable"),
    ("What is the overdraft fee?", "not in the policy"),
    ("Is there a fee for paying with Apple Pay?", "not in the policy"),
]

In [ ]:
def grounded_prompt(question, rule):
    system = "Answer the customer's question using only the bank policy below.\n\n"
    system += policy

    if rule:
        system += "\n\nIf the policy does not contain the answer, reply exactly: I don't know"

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": question}
    ]

# test: one unanswerable question with the rule on
if grounded_prompt(questions[5][0], True) is None:
    print("Write grounded_prompt above, then run this cell again.")
else:
    print("Q:", questions[5][0])
    print("A:", ask(grounded_prompt(questions[5][0], True), max_new_tokens=80))


for rule in [False, True]:
    if rule:
        print("===== rule on =====")
    else:
        print("===== rule off =====")
    for question, kind in questions:
        print("Q:", question, "  [" + kind + "]")
        # TODO: ask the question with grounded_prompt, max_new_tokens=80
        print("A:", ask(grounded_prompt(question, rule), max_new_tokens=80))
        print()

# Expected output (your replies will differ):
#
# ===== rule off =====
# Q: How much does a card payment in euros cost?   [answerable]
# A: ...
# [all six questions]
#
# ===== rule on =====
# [all six questions]

Q: Is there a fee for paying with Apple Pay?
A: No, there is no fee for paying with Apple Pay.
===== rule off =====
Q: How much does a card payment in euros cost?   [answerable]
A: A card payment in euros would cost 2% of the payment amount.

Q: I already used the ATM three times this month. What will the next withdrawal cost?   [answerable]
A: Each subsequent ATM withdrawal after the first three will cost $2.50.

Q: Does it cost anything to top up with a credit card?   [answerable]
A: Yes, topping up with a credit card costs 1.5%.

Q: My card was stolen this morning. Do I pay for the new card?   [answerable]
A: Yes, you will need to pay $5 for a replacement card as per your bank policy.

Q: What is the overdraft fee?   [not in the policy]
A: There is no specific mention of an overdraft fee in the provided Northside Bank fee policy. If you experience an overdraft on your account, please contact our customer service for assistance.

Q: Is there a fee for paying with Apple Pay?   [not in

## Use of AI


<font color='red'> I acknowledge the usage of AI in the Module 7 Challenege Lab in the form of LLM model Google Gemini. Google Gemini is a built in feature in Google Colab that can help identify errors in your code, autofill in code using context from other code cells, and suggest code improvement. The built in code fixing and autofill functions were used throughout the lab in multiple code blocks to fix errors and fill in commands.

<font color='red'>I acknowledge the use of AI in the Module 7 Challenge Lab in the form of LLM model ChatGPT. ChatGPT was used in code block 11 to generate a full_prompt(message) and reduced_prompt(message).

<font color='red'>I acknowledge the use of AI in the Module 7 Challenge Lab in the form of LLM model ChatGPT. ChatGPT was used in code block 15 to generate a check for the code block above.

<font color='red'>I acknowledge the use of AI in the Module 7 Challenge Lab in the form of LLM model ChatGPT. ChatGPT was used in code block 17 to generate example_question and example_answer.

<font color='red'> I acknowledge the use of AI in the Module 7 Challenge Lab in the form of LLM model ChatGPT. ChatGPT was used in code block 19 to generate the ticket_prompt(message).

## References

- Guided Lab Part 1, Building Prompts, and Guided Lab Part 2, Reasoning and Structured Output, Module 7
- Alammar and Grootendorst (2024), *Hands-On Large Language Models*, Chapter 6, Prompt Engineering
- Model card: [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- Dataset card: [BANKING77](https://huggingface.co/datasets/legacy-datasets/banking77); Casanueva et al. (2020), [Efficient Intent Detection with Dual Sentence Encoders](https://arxiv.org/abs/2003.04807)
- Dataset card: [GSM8K](https://huggingface.co/datasets/openai/gsm8k); Cobbe et al. (2021), [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168)
- Wei et al. (2022), [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)
- Kojima et al. (2022), [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/abs/2205.11916)